# 🎮 Aprendizaje por Refuerzo con `harness_rl.py`

> **Objetivo:** entender el gradiente de política **de verdad** — no la metáfora del
> agente y la zanahoria, sino de dónde sale cada número que entra en `loss.backward()`.

---

## La única idea que hay que llevarse

Todo este notebook desarrolla una sola frase:

> **El gradiente de política es aprendizaje supervisado sobre tus propias muestras,
> ponderado por lo bien que salieron.**

No es un algoritmo nuevo. Es la entropía cruzada de toda la vida con **dos cambios**:

| | Supervisado | Refuerzo |
|---|---|---|
| ¿Quién pone la diana? | Un **anotador humano** | **El propio modelo** (la acción que eligió) |
| ¿Cuánto pesa cada muestra? | Siempre **1** | La **recompensa** obtenida |

Todo lo demás —retornos, ventajas, GAE, PPO— son técnicas para que ese
estimador tenga menos varianza. La fórmula no cambia.

---

## El recorrido

```none
1. La idea en 6 líneas        ── las dos pérdidas, lado a lado, con números
2. Radiografía de un rollout  ── ¿de dónde salen los datos? (la diferencia real)
3. Reparto de mérito          ── recompensas ──► retornos hacia adelante
4. ¿Bueno comparado con qué?  ── los 3 modos de ponderación y el cambio de signo
5. El bandido de dos brazos   ── el "hola mundo": ¿qué palanca paga más?
6. El corredor y gamma        ── mismo entorno, gamma distinto, decisión OPUESTA
7. La trampa de la varianza   ── por qué una semilla NO es un resultado
8. La loss no es la métrica   ── la pérdida sube mientras el agente mejora
9. Puente a RLHF              ── cuando el juez es una red: reward hacking en vivo
```

---

> ⚠️ **Lo que este notebook NO es.** `harness_rl.py` implementa REINFORCE y nada más:
> sin crítico $V(s)$, sin GAE, sin PPO, sin replay buffer. Es la versión más pequeña que
> sigue siendo honesta, elegida para que se pueda leer entera. Para RL de verdad se usa
> `CleanRL` (para entender), `Stable-Baselines3` (para usar) o `TRL` (para alinear LLMs).
> El mapa completo de lo que falta está al final de `lab/harness_rl.py`, sección 🔀.

## 0. Arranque

`harness_rl.py` importa `harness.py` entero en vez de copiar piezas: si `save_run` mejora
allí, mejora aquí gratis. También registra `adam` y `sgd` por su cuenta (sin sobrescribir
los tuyos si ya los tenías), así que no hace falta prepararle nada.

In [ ]:
# ── ARRANQUE ──
import os, sys
from pathlib import Path

while not (Path.cwd() / "lab").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch.distributions import Categorical

from lab import harness as H
from lab import harness_rl as rl

CPU = torch.device("cpu")

print("✅ Arnés de refuerzo listo en:", Path.cwd())
print("   Entornos registrados     :", sorted(rl.envs))
print("   Políticas registradas    :", [k for k in sorted(H.models) if "policy" in k])
print("   Optimizadores            :", sorted(H.optimizers))

---
## 1. La idea entera, en seis líneas

Vamos a calcular las dos pérdidas sobre **los mismos logits** y comprobar que son la misma
operación.

Imagina una red que ante cierta situación produce tres números (logits) para tres opciones.
En clasificación las llamaríamos *clases*; en refuerzo las llamamos *acciones*. Son los
mismos números.

In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0]])   # 3 opciones. ¿Clases o acciones? Las mismas.

# ── A. SUPERVISADO ── la diana la puso un humano, y pesa 1
etiqueta = torch.tensor([1])                        # "la correcta es la opción 1"
loss_sup = F.cross_entropy(logits, etiqueta)

# ── B. REFUERZO ── la diana la eligió el modelo, y pesa la recompensa
dist    = Categorical(logits=logits)                # aplica Softmax internamente
accion  = torch.tensor([1])                         # el modelo muestreó la opción 1
log_p   = dist.log_prob(accion)                     # log π(a|s)

print("Probabilidades del modelo:", dist.probs.numpy().round(3))
print()
print(f"  cross_entropy(logits, etiqueta)  = {loss_sup.item():+.6f}")
print(f"  -log_prob(accion)                = {-log_p.item():+.6f}")
print(f"  ¿Son el mismo número?            → {torch.allclose(loss_sup, -log_p)}")

**Son literalmente la misma operación.** `F.cross_entropy(logits, y)` es
$-\log \pi(y \mid s)$, y punto.

Entonces, ¿dónde está el refuerzo? En **multiplicar por la recompensa**. Ese `* R` es todo
lo que separa los dos paradigmas — y fíjate en lo que hace el **signo**:

In [ ]:
print("  Recompensa │   Pérdida   │ ¿Qué le pasa a la probabilidad de esa acción?")
print("  ───────────┼─────────────┼──────────────────────────────────────────────")
for R in (+1.0, +0.3, 0.0, -1.0):
    loss_rl = -(log_p * R)
    if   R > 0: efecto = "SUBE  → 'haz más de esto'"
    elif R == 0: efecto = "no se mueve → gradiente nulo"
    else:        efecto = "BAJA  → 'haz menos de esto'"
    print(f"    {R:+.1f}     │  {loss_rl.item():+.6f}  │ {efecto}")

print()
print("La recompensa NEGATIVA invierte el signo del gradiente. Eso es todo el mecanismo:")
print("minimizar -log(p)*R con R>0 empuja p hacia arriba; con R<0, hacia abajo.")

> 🔑 **Una consecuencia poco intuitiva:** si **todas** tus recompensas son positivas,
> **todas** las acciones se refuerzan (unas más que otras). El algoritmo aprende igual,
> porque lo que importa es la diferencia *relativa*, pero lo hace despacio y con mucho
> ruido. Arreglar eso es justo lo que hace la sección 4.

---
## 2. Radiografía de un rollout: ¿de dónde salen los datos?

Aquí está la diferencia **estructural** con `harness.py`, y no es la pérdida: es que en
refuerzo **los datos no existen hasta que el modelo los fabrica**.

```none
harness.py     :  datasets.build()  ──► corre UNA vez, antes del bucle
harness_rl.py  :  collect_episode() ──► corre en CADA iteración, y usa el modelo
```

Vamos a fabricar un episodio a mano y abrirlo por dentro. Usamos el entorno `corridor`:
la política empieza en la posición 0 y en cada paso decide **cobrar y salir** (izquierda,
`+0.3` seguro) o **seguir** (derecha, cuesta `-0.05` por paso pero paga `+1.0` al llegar).

In [ ]:
env      = rl.envs.build("corridor", length=5, step_cost=0.05, quit_reward=0.3)
politica = H.models.build("mlp_policy", obs_dim=env.obs_dim,
                          n_actions=env.n_actions, hidden=32)

H.set_seed(0)
tray = rl.collect_episode(politica, env, CPU, max_steps=50)

print(f"Episodio de {len(tray)} pasos · recompensa total {tray.total_reward:+.3f}")
print()
print("  paso │  recompensa │  log π(a|s) │  entropía │ ¿lleva grafo de gradiente?")
print("  ─────┼─────────────┼─────────────┼───────────┼──────────────────────────")
for t, (r, lp, ent) in enumerate(zip(tray.rewards, tray.log_probs, tray.entropies)):
    print(f"   {t}   │   {r:+.3f}    │   {lp.item():+.4f}   │   {ent.item():.3f}   │"
          f"  log_p: {lp.requires_grad}   r: (float puro)")

Fíjate en la última columna, porque es donde está todo el truco de la implementación:

* **`log_probs` llevan grafo de gradiente** (`requires_grad=True`). Salieron de la red, así
  que `backward()` puede viajar por ellos hasta los pesos. **Son el único camino del
  gradiente.**
* **`rewards` son floats de Python.** No hay grafo, no hay derivada. Son un *veredicto*
  sobre lo que pasó, no una cantidad a optimizar.

> ⚠️ Esto es la fuente del error más caro de RLHF: si tu recompensa viene de un modelo
> (un clasificador), **tienes que envolverla en `torch.no_grad()`**. Si no, el gradiente se
> escapa hacia el juez e intentas "mejorar" al examinador en vez de al alumno. En
> `harness_rl.py` esto se blinda con un `weights.detach()` explícito.

Y ahora la propiedad incómoda: **estos datos son de un solo uso.**

In [ ]:
# Un paso de optimizador cualquiera...
opt = torch.optim.SGD(politica.parameters(), lr=0.5)
antes = politica(env.reset()).detach().clone()
loss = -(torch.stack(tray.log_probs) * 1.0).mean()
opt.zero_grad(); loss.backward(); opt.step()
despues = politica(env.reset()).detach()

print("Logits de la política ante el estado inicial:")
print("  antes del paso :", antes.numpy().round(4))
print("  después        :", despues.numpy().round(4))
print()
print("→ La política ha CAMBIADO. La trayectoria de arriba la generó una política")
print("  que ya no existe, así que describe a alguien que ya no está. Se tira.")
print()
print("  Eso es lo que significa 'on-policy', y es la ineficiencia fundamental del")
print("  gradiente de política: cada dato se usa UNA vez. Los métodos off-policy (DQN,")
print("  SAC) existen precisamente para poder reciclar experiencia vieja.")

---
## 3. Reparto de mérito: de recompensas a retornos

Problema: en el corredor la recompensa buena llega **al final**. Los cuatro pasos previos
solo cuestan `-0.05`. Si juzgásemos cada paso por su recompensa inmediata, concluiríamos
que caminar es malo — y nunca aprenderíamos a llegar.

La solución es juzgar cada paso **por su futuro**, no por su presente:

$$G_t = r_t + \gamma \, r_{t+1} + \gamma^2 r_{t+2} + \dots$$

`returns_to_go()` recorre las recompensas **hacia atrás** precisamente por eso: el mérito
de un paso depende de lo que vino después.

In [ ]:
# El episodio "paciente": 4 pasos que solo cuestan, y el premio al final
recompensas = [-0.05, -0.05, -0.05, -0.05, +0.95]

print("  Recompensas inmediatas :", [f"{r:+.2f}" for r in recompensas])
print()
print("  gamma │  G_0     G_1     G_2     G_3     G_4   │ ¿Qué dice sobre el paso 0?")
print("  ──────┼──────────────────────────────────────────┼──────────────────────────")
for gamma in (1.0, 0.99, 0.9, 0.5):
    G = rl.returns_to_go(recompensas, gamma)
    veredicto = "vale la pena caminar" if G[0] > 0.3 else "NO vale la pena (mejor cobrar)"
    print(f"   {gamma:.2f} │ " + "  ".join(f"{g:+.3f}" for g in G) + f" │ {veredicto}")

Lee la primera columna (`G_0`, el juicio sobre el primer paso) de arriba abajo. **El mismo
episodio, exactamente las mismas recompensas, y el veredicto se invierte.**

Eso es lo que significa `gamma`: no es un hiperparámetro que se ajusta buscando el mejor
número, es **parte de la definición del problema**. Estás declarando cuánta paciencia tiene
tu agente. Con `gamma=0.5` el premio de dentro de 4 pasos vale la dieciseisava parte, y
renunciar a él es *racional*.

> 💡 En episodios cortos gamma apenas se nota. En episodios largos es el hiperparámetro que
> más duele equivocar, porque no te da un error: te da un agente que resuelve un problema
> distinto del que creías haber planteado.

---
## 4. "¿Bueno comparado con qué?" — los tres modos de ponderación

Ya tenemos un número por paso ($G_t$). Podríamos usarlo directamente como peso... y
funciona mal. El problema es la pregunta implícita: `+0.75` **¿es bueno?**

No se puede saber sin un punto de referencia. `compute_weights()` ofrece tres respuestas,
que son los tres primeros escalones de una escalera muy larga.

Preparamos un lote realista: **un episodio paciente** (5 pasos, llega al premio) y **un
episodio cobarde** (1 paso, cobra `+0.3` y se va).

In [ ]:
returns = torch.tensor(rl.returns_to_go([-0.05]*4 + [0.95], gamma=0.99) + [0.30])
etiquetas = ["paciente t0", "paciente t1", "paciente t2", "paciente t3",
             "paciente t4", "COBARDE t0"]

print("  Paso          │  retorno │  'return'  │ 'baseline' │ 'normalized'")
print("  ──────────────┼──────────┼────────────┼────────────┼─────────────")
modos = {m: rl.compute_weights(returns, m) for m in ("return", "baseline", "normalized")}
for i, nombre in enumerate(etiquetas):
    fila = " │ ".join(f"{modos[m][i].item():+9.4f}" for m in modos)
    print(f"  {nombre:13s} │ {returns[i].item():+8.4f} │ {fila}")

print()
print(f"  media de los retornos = {returns.mean().item():+.4f}")

Mira la fila **`COBARDE t0`** al cambiar de columna:

* **`return`** → peso `+0.30`. Positivo, así que **cobrar se refuerza**. El algoritmo le
  dice a la política "haz más de esto", cuando es justo la acción que queremos evitar.
  Aprende igualmente (porque los pesos de los pasos pacientes son mayores), pero está
  remando a favor y en contra a la vez.
* **`baseline`** → peso **negativo**. Ahora sí: al restar la media, "bueno" pasa a
  significar **"mejor que el promedio"**, y cobrar queda por debajo. La acción se
  desalienta de verdad. Y esto **no sesga** el gradiente: restar una constante es
  matemáticamente gratis.
* **`normalized`** → lo mismo, pero además dividido por la desviación, dejando los pesos
  en torno a ±1. Desacopla la tasa de aprendizaje de la escala de las recompensas (da igual
  si tu juego puntúa de 0 a 1 o de 0 a 10.000). Técnicamente **sí** introduce un pequeño
  sesgo; todo el mundo lo usa igualmente porque en la práctica funciona.

Y un matiz que se ve en la tabla y conviene no pasar por alto: **`paciente t0` también
queda ligeramente negativo**. Tiene sentido — está por debajo de la media del lote, porque
es el paso más lejano del premio y arrastra todos los costes. El algoritmo lo penaliza un
poco, aunque formara parte del episodio bueno.

Con solo dos episodios en el lote eso es ruido, y es precisamente el síntoma de que restar
la media del lote es una baseline **pobre**: mezcla en un mismo promedio estados que no son
comparables (empezar el corredor y estar a un paso de la meta no valen lo mismo). La cura
es el siguiente escalón.

> 📈 **Los escalones siguientes** (que `harness_rl.py` deliberadamente no tiene): en lugar
> de restar la media del lote, entrenar **una segunda red $V(s)$** que prediga "cuánto
> esperaba sacar desde este estado". Eso convierte REINFORCE en **Actor-Crítico**, y es el
> salto de juguete a real. Luego **GAE** para afinar el compromiso sesgo/varianza, y **PPO**
> para impedir que un solo paso destruya la política.

---
## 5. Experimento 1: el bandido de dos brazos

El "hola mundo" del refuerzo. Un estado (constante), dos acciones, y el episodio dura
**un paso**. No hay tiempo, no hay crédito que repartir: la política solo tiene que
descubrir cuál de las dos palancas paga más.

La palanca derecha paga con probabilidad **0.8**; la izquierda, **0.2**. Nadie se lo dice:
tiene que averiguarlo tirando.

Sirve como **prueba de humo**: si esto no converge a ~0.8, el bug está en el gradiente y no
en la tarea.

In [ ]:
print("Config de partida que trae el arnés:")
for k, v in rl.BANDIT_CONFIG.items():
    print(f"   {k:24s} = {v}")

In [ ]:
res_bandido = rl.run_rl_experiment(rl.BANDIT_CONFIG)
print(f"\n💾 Guardado en runs/{res_bandido.run_id}/")

Ahora la pregunta que de verdad importa: **¿qué ha aprendido?** En un problema tan pequeño
podemos abrir la política y leerle el pensamiento directamente.

In [ ]:
with torch.no_grad():
    probs = Categorical(logits=res_bandido.model(torch.zeros(1))).probs

print("Lo que cree la política sobre qué palanca tirar:")
print(f"   P(izquierda) = {probs[0]:.4f}   (paga 0.2 de las veces)")
print(f"   P(derecha)   = {probs[1]:.4f}   (paga 0.8 de las veces)")
print()
ultimas = np.mean([h["reward_mean"] for h in res_bandido.history[-10:]])
print(f"Recompensa media (últimas 10 iteraciones): {ultimas:+.3f}")
print(f"Óptimo teórico alcanzable                : +0.800")
print()
print("Ojo a una cosa: alguna iteración de arriba SUPERA el 0.800. No es magia ni un")
print("bug — son 32 tiradas de una moneda sesgada, y la media de una muestra pequeña")
print("fluctúa por encima y por debajo. Es exactamente el ruido que hace que en RL")
print("una sola medición no signifique nada (sección 7).")
print()
print("→ La política no ha aprendido las probabilidades 0.2/0.8: ha aprendido a tirar")
print("  SIEMPRE de la buena. Y eso es correcto. No está modelando el mundo, está")
print("  optimizando el comportamiento. Es la diferencia entre predecir y decidir.")

In [ ]:
hist = H.load_run(res_bandido.run_id)["history"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(hist["epoch"], hist["reward_mean"], label="entrenamiento (explora)", lw=2)
ax1.plot(hist["epoch"], hist["eval_reward"], label="evaluación (argmax)", lw=2, ls="--")
ax1.axhline(0.8, color="grey", ls=":", label="óptimo teórico (0.8)")
ax1.set_xlabel("iteración"); ax1.set_ylabel("recompensa media")
ax1.set_title("Aprende a tirar de la palanca buena")
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

ax2.plot(hist["epoch"], hist["entropy"], color="crimson", lw=2)
ax2.set_xlabel("iteración"); ax2.set_ylabel("entropía de la política")
ax2.set_title("La exploración se apaga (y no vuelve)")
ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

El panel derecho es el que hay que mirar con cuidado. La **entropía** mide cuánto reparte
la política sus probabilidades, o sea, **cuánto explora**. Empieza alta (duda) y se hunde
a cero (ha decidido).

En el bandido eso está bien: ha decidido lo correcto. Pero fíjate en que **el proceso es
irreversible**: con entropía 0 la política ya no muestrea nada nuevo, así que si el mundo
cambiara —o si simplemente se hubiera equivocado— no hay forma de que se entere.

> 🎯 Ese es el **dilema exploración/explotación**, y aquí lo estás viendo en un gráfico. El
> parámetro `entropy_coef` de `harness_rl.py` es un parche contra esto: paga a la política
> por seguir dudando. En la sección 7 veremos que el parche no basta.

---
## 6. Experimento 2: el corredor, o el poder de `gamma`

Ahora el entorno con dilema. En cada paso la política elige:

* **cobrar y salir** (izquierda) → `+0.30` seguro, episodio terminado.
* **seguir** (derecha) → `-0.05` por paso, y `+1.00` si llega al final (5 pasos).

Aguantar renta `1.00 - 5×0.05 = +0.75`, más del doble que cobrar. Pero hay que **esperar**.

Vamos a lanzar el **mismo config dos veces**, cambiando **solo `gamma`**.

In [ ]:
resultados = {}
for gamma in (0.99, 0.60):
    cfg = dict(rl.CORRIDOR_CONFIG, gamma=gamma, name=f"corredor_g{gamma}")
    print(f"▶ gamma = {gamma}")
    resultados[gamma] = rl.run_rl_experiment(cfg, verbose=False)

print()
print("  gamma │ recompensa │ pasos/episodio │ política aprendida")
print("  ──────┼────────────┼────────────────┼───────────────────────────────")
for gamma, res in resultados.items():
    h = res.history[-1]
    decision = "AGUANTA hasta el premio" if h["episode_len"] > 2 else "COBRA YA y se va"
    print(f"   {gamma:.2f} │   {h['reward_mean']:+.3f}   │      {h['episode_len']:.2f}      │ {decision}")

**El mismo entorno. El mismo código. La misma arquitectura. Decisiones opuestas.**

Con `gamma=0.99` el premio de dentro de 5 pasos casi no se descuenta y aguantar es lo
racional. Con `gamma=0.60` ese premio vale `0.6⁴ ≈ 0.13`, menos que el `+0.30` de cobrar
ya — así que la política aprende, **correctamente**, a rendirse.

> ⚠️ Ninguna de las dos está mal entrenada. Están resolviendo **dos problemas distintos**,
> y la única diferencia es un número en el config. Cuando en RL alguien dice "el agente
> hace algo raro", muy a menudo el agente está resolviendo perfectamente un problema que
> no era el que se pretendía plantear.

Lo mismo, en curvas:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
colores = {0.99: "seagreen", 0.60: "indianred"}
for gamma, res in resultados.items():
    h = H.load_run(res.run_id)["history"]
    ax.plot(h["epoch"], h["reward_mean"], lw=2, color=colores[gamma],
            label=f"gamma = {gamma}")

ax.axhline(0.75, color="seagreen", ls=":", alpha=0.7, label="aguantar = +0.75")
ax.axhline(0.30, color="indianred", ls=":", alpha=0.7, label="cobrar ya = +0.30")
ax.set_xlabel("iteración"); ax.set_ylabel("recompensa media")
ax.set_title("El mismo entorno, dos definiciones distintas de 'bueno'")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 7. La trampa: una semilla no es un resultado

Esta sección es la más importante del notebook, y la que más gente se salta.

En aprendizaje supervisado, repetir un experimento con varias semillas es **buena
práctica**. En refuerzo es **la diferencia entre un resultado y una anécdota**: el
estimador de REINFORCE tiene varianza tan alta que dos semillas del mismo config pueden
dar respuestas contrarias.

Lanzamos **cinco veces el mismo config** del corredor con `gamma=0.99` — donde ya hemos
visto que aguantar es lo óptimo.

In [ ]:
tabla = rl.repeat_rl_with_seeds(dict(rl.CORRIDOR_CONFIG, name="corredor_semillas"),
                               n_seeds=5)
print()
print(tabla[["run_id", "seed", "gamma", "final", "best"]].to_string(index=False))

Ahí lo tienes. **El mismo config, el mismo código, la misma tarea**: unas semillas aprenden
a aguantar (`+0.75`) y otras se atascan cobrando (`+0.30`).

No es un bug. Es el **óptimo local** clásico del gradiente de política, y la secuencia es
siempre la misma:

```none
  1. Las primeras trayectorias son aleatorias.
  2. Por azar, alguna cobra pronto y se lleva su +0.30.
  3. Esa acción se refuerza. Su probabilidad sube.
  4. La entropía se hunde → la política deja de explorar.
  5. Nunca llega a ver el +1.00 del final. Ya no puede aprender que existía.
```

Ese `entropy_coef: 0.01` del config está puesto exactamente contra esto, y **sube el suelo
pero no lo arregla**. Lo que lo arregla de verdad es lo que este arnés no tiene: una
baseline aprendida $V(s)$ y PPO.

> 🔬 **La lección metodológica:** si hubieras lanzado una sola semilla, tenías un 60% de
> probabilidad de escribir *"funciona"* y un 40% de escribir *"no funciona"*. Las dos
> conclusiones habrían sido falsas. El aviso que imprime el arnés
> (*"una diferencia menor que ~2σ NO es un resultado"*) es literal.

Vamos a dibujar el abanico, porque verlo convence más que leerlo:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for run_id in tabla["run_id"]:
    h = H.load_run(run_id)["history"]
    semilla = H.load_run(run_id)["meta"]["seed"]
    gano = h["eval_reward"].iloc[-1] > 0.5
    ax.plot(h["epoch"], h["eval_reward"], lw=1.8, alpha=0.85,
            color="seagreen" if gano else "indianred",
            label=f"semilla {semilla} → {'aguanta' if gano else 'cobra'}")

ax.set_xlabel("iteración"); ax.set_ylabel("recompensa (política determinista)")
ax.set_title("Cinco veces el MISMO experimento")
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("Si promediaras estas cinco curvas en una sola línea, la media diría ~+0.57:")
print("una recompensa que NINGUNA de las cinco ejecuciones obtuvo jamás.")

---
## 8. Por qué la `loss` no te sirve de métrica

En `harness.py` todo el flujo de trabajo se apoya en una idea: **`val_loss` baja = vamos
bien**. En refuerzo esa idea **no se sostiene**, y conviene verlo con los ojos.

In [ ]:
h = H.load_run(res_bandido.run_id)["history"]

fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(h["epoch"], h["policy_loss"], color="slateblue", lw=2)
ax1.set_xlabel("iteración")
ax1.set_ylabel("pérdida de política", color="slateblue")
ax1.tick_params(axis="y", labelcolor="slateblue")
ax1.axhline(0, color="slateblue", ls=":", alpha=0.4)

ax2 = ax1.twinx()
ax2.plot(h["epoch"], h["reward_mean"], color="darkorange", lw=2)
ax2.set_ylabel("recompensa media (lo que de verdad importa)", color="darkorange")
ax2.tick_params(axis="y", labelcolor="darkorange")

ax1.set_title("La pérdida no mide el progreso")
plt.tight_layout(); plt.show()

print(f"pérdida inicial {h['policy_loss'].iloc[0]:+.4f} → final {h['policy_loss'].iloc[-1]:+.4f}")
print(f"recompensa      {h['reward_mean'].iloc[0]:+.4f} → final {h['reward_mean'].iloc[-1]:+.4f}")

La pérdida de política se va a **cero** — y no porque el agente sea perfecto, sino porque
`log π(a|s) → 0` cuando la política se vuelve determinista. Es un número que habla de
**cuánto está cambiando** la política, no de **lo bien que lo hace**.

Puede subir, bajar, oscilar o quedarse plana mientras el agente mejora. La única métrica
que significa algo es la **recompensa**.

Y esto tiene una consecuencia práctica en el arnés:

```python
H.compare_runs(ids, "reward_mean")   # ⚠️ calcula 'best' con .min()  → la PEOR iteración
rl.compare_rl_runs(ids, "reward_mean")  # ✅ calcula 'best' con .max()
```

`H.compare_runs` no da error: te devuelve la peor iteración etiquetada como `best`, que es
bastante peor que un error. Por eso `harness_rl.py` trae su propia versión.

In [ ]:
ids = [res_bandido.run_id]
print("Con el helper supervisado (best = mínimo, INCORRECTO aquí):")
print(H.compare_runs(ids, "reward_mean")[["run_id", "final", "best"]].to_string(index=False))
print()
print("Con el helper de refuerzo (best = máximo):")
print(rl.compare_rl_runs(ids, "reward_mean")[["run_id", "final", "best"]].to_string(index=False))

---
## 9. El puente a RLHF: cuando el juez es una red

Hasta aquí la recompensa la daba el entorno, gratis y objetiva. En un modelo de lenguaje
**eso no existe**: no hay ninguna función matemática que puntúe si una frase es buena.

La solución de RLHF es la que ya conoces del notebook anterior:

```none
FASE 1 ──► LM preentrenado ──────────────────┐
   (predice siguiente palabra)               │
                                             ▼
FASE 2 ──► Clasificador de sentimiento ──► ES EL REWARD MODEL
   (pocos ejemplos etiquetados)              │
                                             │ puntúa
                                             ▼
FASE 3 ──► El LM de Fase 1 GENERA texto ──► recompensa ──► REINFORCE
   (política = LM, acción = token)
```

Vamos a montar la versión de juguete: un vocabulario diminuto, un "juez" escrito a mano
(en RLHF real sería el clasificador de la Fase 2) y una política que emite 4 palabras.

**Fíjate en que el entorno se registra AQUÍ, en el notebook, no en `harness_rl.py`.** Es
deliberado: depende del vocabulario concreto de este problema. Mismo patrón que los
datasets del arnés supervisado — la librería trae el contrato, el notebook trae el problema.

In [ ]:
VOCAB     = ["el", "servicio", "es", "excelente", "malo", "pesimo", "maravilla"]
POSITIVAS = {"excelente", "maravilla"}
LONGITUD  = 4


def juez(tokens):
    '''El reward model de juguete: fracción de palabras positivas.

    En RLHF real esto es un CLASIFICADOR entrenado con preferencias humanas, y
    se le llama con torch.no_grad() porque es un juez, no un alumno.
    '''
    palabras = [VOCAB[t] for t in tokens]
    return sum(1.0 for p in palabras if p in POSITIVAS) / len(palabras)


@rl.envs.register("critica_puntuada")
def build_critica(longitud=LONGITUD, **kwargs):
    class Critica:
        obs_dim, n_actions = 1, len(VOCAB)

        def reset(self):
            self.tokens = []
            return self._observe()

        def _observe(self):
            # Solo ve en qué posición de la frase está. Una política de verdad
            # vería las palabras anteriores (eso es lo que hace un LM).
            return torch.tensor([len(self.tokens) / longitud], dtype=torch.float32)

        def step(self, action):
            self.tokens.append(action)          # la ACCIÓN es emitir un token
            done = len(self.tokens) >= longitud
            reward = juez(self.tokens) if done else 0.0   # el juez habla al final
            return self._observe(), reward, done

    return Critica()


print("✅ Entorno 'critica_puntuada' registrado desde el notebook")
print("   Vocabulario :", VOCAB)
print("   Positivas   :", sorted(POSITIVAS))

In [ ]:
def escribir(politica, greedy=True):
    '''Genera una frase con la política y la puntúa.'''
    env = rl.envs.build("critica_puntuada")
    rl.collect_episode(politica, env, CPU, max_steps=LONGITUD, greedy=greedy)
    # Los tokens se leen del entorno: Trajectory guarda log_probs y recompensas,
    # pero NO las acciones. Es una limitación real del arnés mínimo.
    return " ".join(VOCAB[t] for t in env.tokens), juez(env.tokens)


H.set_seed(0)
sin_entrenar = H.models.build("mlp_policy", obs_dim=1, n_actions=len(VOCAB), hidden=32)
frase, r = escribir(sin_entrenar)
print(f"ANTES de entrenar : \"{frase}\"   → recompensa {r:.2f}")

In [ ]:
config_texto = {
    "name": "rl_critica",
    "env": "critica_puntuada",
    "model": "mlp_policy",
    "model_args": {"hidden": 32},
    "optimizer": "adam",
    "optimizer_args": {"lr": 0.05},
    "iterations": 60,
    "episodes_per_iteration": 24,
    "gamma": 1.0,
    "advantage": "normalized",
    "entropy_coef": 0.01,
    "max_steps": LONGITUD,
    "eval_episodes": 10,
    "seed": 0,
}

res_texto = rl.run_rl_experiment(config_texto, verbose=False)
frase, r = escribir(res_texto.model)
print(f"DESPUÉS de entrenar: \"{frase}\"   → recompensa {r:.2f}")
print()
print(f"Recompensa media final: {res_texto.history[-1]['reward_mean']:.3f} (máximo posible: 1.000)")

### 🎣 Ahí lo tienes: *reward hacking* en vivo

La política ha conseguido **recompensa máxima** produciendo algo que **no es español**.

Y ha hecho exactamente lo que se le pidió. Nadie le dijo "escribe bien": se le dijo
"maximiza la puntuación del juez", y el juez cuenta palabras positivas. Repetir la misma
palabra positiva cuatro veces es la solución **óptima** al problema planteado.

Esto no es un defecto de este juguete. Es
[la ley de Goodhart](https://en.wikipedia.org/wiki/Goodhart%27s_law) —*cuando una medida se
convierte en objetivo, deja de ser una buena medida*— y es **el** problema central de la
alineación. El clasificador es una **aproximación** al juicio humano, y optimizar con fuerza
contra una aproximación siempre acaba explotando el hueco entre la aproximación y lo
aproximado.

#### El parche estándar: penalización KL

$$R_{\text{final}} = R_{\text{juez}} - \beta \cdot D_{KL}\big(\pi_{\text{actual}} \,\|\, \pi_{\text{Fase 1}}\big)$$

En palabras: *"gana puntos por agradar al juez, pero **pierde puntos por alejarte de cómo
hablabas antes**"*. El modelo preentrenado actúa de ancla lingüística.

Y aquí se cierra el círculo con el notebook anterior: **para poder calcular ese término hace
falta mantener congelada una copia del modelo de la Fase 1**. Por eso la Fase 3 necesita
**tres modelos vivos a la vez**:

| Modelo | Estado | Papel |
|---|---|---|
| Política | entrenable | El que genera y aprende |
| Reward model (Fase 2) | congelado, `eval()` | El juez |
| Referencia (Fase 1) | congelada | El ancla del término KL |

Ese checkpoint inmutable de la Fase 1 no era burocracia de trazabilidad: **es un ingrediente
del algoritmo**.

> 🔧 `harness_rl.py` **no** implementa la penalización KL, y está anotado como tal en su
> mapa 🔀. Para RLHF de verdad, `TRL` (`PPOTrainer`, `DPOTrainer`) lo trae hecho.

---
# 💡 Conclusiones

### Sobre el algoritmo

1. **El gradiente de política es entropía cruzada ponderada.** Lo verificamos con números
   en la sección 1: `F.cross_entropy(logits, y)` *es* $-\log \pi(y|s)$. Todo el refuerzo
   está en el `* R` y en que la diana la elige el modelo.
2. **Los datos son de un solo uso.** Tras `optimizer.step()` la política cambió, así que las
   trayectorias describen a alguien que ya no existe. Eso es *on-policy*, y es la
   ineficiencia fundamental del método.
3. **El gradiente viaja solo por los `log_probs`.** Las recompensas son floats sin grafo. En
   cuanto el juez sea una red, `no_grad()` deja de ser una optimización y pasa a ser
   obligatorio.
4. **`gamma` no es un hiperparámetro, es parte del problema.** Sección 6: el mismo entorno
   con dos gammas produce políticas opuestas, y **las dos son correctas**.
5. **"Bueno" no significa nada sin un punto de comparación.** Restar la baseline es lo que
   convierte "positivo" en "mejor que la media" y permite desalentar acciones de verdad.

### Sobre el método

6. **Una semilla no es un resultado.** 3 de 5 semillas aprendieron; 2 se atascaron. Con una
   sola ejecución tenías un 40% de probabilidad de concluir lo contrario de la verdad.
7. **La `loss` no mide el progreso.** Se va a cero porque la política se vuelve
   determinista, no porque acierte. Mira la recompensa, y usa `compare_rl_runs` (máximo),
   no `compare_runs` (mínimo).
8. **Optimizar contra un juez aprendido produce *reward hacking*.** Siempre. No es un bug
   que se arregle con más épocas: es Goodhart, y se contiene con la penalización KL contra
   el modelo de referencia.

### Lo que NO hemos hecho (y dónde seguir)

| Falta | Por qué importa | Dónde |
|---|---|---|
| Crítico $V(s)$ | El salto de juguete a real | A2C / PPO · CleanRL `ppo.py` |
| GAE | Afina el compromiso sesgo/varianza | [arXiv:1506.02438](https://arxiv.org/abs/1506.02438) |
| PPO | Evita que un paso destruya la política | [arXiv:1707.06347](https://arxiv.org/abs/1707.06347) |
| Replay buffer | Reciclar experiencia (off-policy) | DQN, SAC |
| Penalización KL | Sin ella, RLHF degenera | TRL `PPOTrainer` |
| DPO | Colapsa reward model + RL en un bucle supervisado | [arXiv:2305.18290](https://arxiv.org/abs/2305.18290) |

**Qué librería según lo que quieras:**

* **Entender** un algoritmo → `CleanRL` (un archivo por algoritmo, sin capas de abstracción)
* **Usarlo** y que funcione → `Stable-Baselines3`
* **Componer** piezas propias → `TorchRL`
* **Alinear un LLM** → `TRL`

---

> 📖 **Para seguir:** el mapa completo de bifurcaciones de diseño está en el propio
> `lab/harness_rl.py` (marcadores `# ALTERNATIVA:` y sección `🔀` al final). El análisis de
> qué exactamente rompe el arnés supervisado está en `lab/HARNESS.md` §4. El desarrollo
> conceptual del pipeline de tres etapas, en
> `docs/01-fundamentos/03-como-se-entrena-una-red/3.7-transferencia-y-finetuning.md`.